<a href="https://colab.research.google.com/github/Darth-Vallabh/Assignment-2/blob/main/ABMWealthDevelopment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Basic ABM Visualization

In [1]:
import numpy as np
import matplotlib.pyplot as plt

import pandas as pd

#Helper Function

# Helper functions

def gini_coefficient(wealth):
    sorted_wealth = np.sort(wealth)
    n = len(wealth)
    cumulative_wealth = np.cumsum(sorted_wealth)
    relative_wealth = cumulative_wealth / cumulative_wealth[-1]
    gini = 1 - 2 * np.sum(relative_wealth) / n + (1 / n)
    return gini

#def initialize_wealth(num_agents):
   # return np.random.uniform(1, 100, num_agents)

def apply_wealth_tax(wealth, wealth_tax_rate):
    return wealth * (1 - wealth_tax_rate)

def apply_inheritance_tax(wealth, inheritance_tax_rate):
    return wealth * (1 - inheritance_tax_rate)

# Function to calculate wealth distribution by decile
def calculate_wealth_distribution(wealth):
    if len(wealth) == 0:
        return [0] * 10
    sorted_wealth = np.sort(wealth)
    total_wealth = np.sum(sorted_wealth)
    decile_size = len(sorted_wealth) // 10
    deciles = []
    for i in range(10):
        start_idx = i * decile_size
        end_idx = (i + 1) * decile_size if i < 9 else len(sorted_wealth)
        decile_wealth = np.sum(sorted_wealth[start_idx:end_idx])
        deciles.append(decile_wealth / total_wealth)
    return deciles

In [2]:
def intialize_wealth(num_agents):
  abc =0

Creating The dummy data

Parameters


In [3]:
num_decile = 10 #input("Num Deciles: ")
wealth_per_decile = [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000] #May need to adjust to more accurate figure displaying wealth per decile based on gini coeffient
birth_rate = [ 0.1, 0.08, 0.07 , 0.065 , 0.05, 0.048 , 0.04 , 0.03, 0.022, 0.2] # Adjust to country birth per decile
death_rate = [0.025, 0.02, 0.018, 0.014, 0.012, 0.011, 0.008, 0.007, 0.006, 0.0005]# Adjust to country death per decile
net_migration = [0.05, 0.04, 0.035, 0.0325, 0.25, 0.024, 0.02, 0.015, 0.011,0.01] #Adjust to country to country migration per decile
Total_Population = 1000 #Number of Agents
Wealth_Tax = 0.20
Savings_Rate = 0.25
num_time_steps = 500
inheritance_tax_rate = 0.25
alpha = 2.00
x_min = 1

Simulation Function considerent Parameters


In [4]:
import pandas as pd

def initialize_wealth(num_agents, wealth_per_decile, birth_rate, death_rate, net_migration):
    """
    Initializes wealth for agents based on their assigned decile.

    Parameters:
    - num_agents (int): Total number of agents.
    - wealth_per_decile (list): List of average wealth values for each decile.
    - birth_rate (list): Birth rates per decile.
    - death_rate (list): Death rates per decile.
    - net_migration (list): Net migration per decile.

    Returns:
    - DataFrame with 'ID', 'Decile', 'Wealth', 'Initial Wealth (T=0)', 'Birth Rate', 'Death Rate', 'Net Migration'.
    """
    agents_per_decile = num_agents // 10  # Equal distribution among deciles
    agent_data = []

    agent_id = 1
    for decile in range(10):  # Deciles are indexed from 0 to 9
        avg_wealth = wealth_per_decile[decile]
        b_rate = birth_rate[decile]
        d_rate = death_rate[decile]
        migration = net_migration[decile]

        for _ in range(agents_per_decile):
            agent_data.append([agent_id, decile + 1, avg_wealth, avg_wealth, b_rate, d_rate, migration])
            agent_id += 1

    # Convert to DataFrame
    df = pd.DataFrame(agent_data, columns=["ID", "Decile", "Wealth", "Initial Wealth (T=0)", "Birth Rate", "Death Rate", "Net Migration"])
    return df

In [5]:
def simulate(initial_num_agents, num_time_steps, wealth_tax_rate, alpha, x_min,
             population_growth_rate, death_rate, inheritance_tax_rate):

    agents = initialize_wealth(initial_num_agents, alpha, x_min)


    gini_coefficients = []
    top_10_share = []
    population_sizes = [initial_num_agents]
    death_counts = []
    birth_counts = []
    inheritance_received = []
    gdp_per_capita = []
    wealth_distribution_by_decile = []
    growth_rates = []
    death_rates_recorded = []

    for t in range(num_time_steps):
        # Apply wealth tax
        agents = apply_wealth_tax(agents, wealth_tax_rate)

        # Random economic growth (normally distributed with mean=0 and std=5)
        growth = np.random.normal(0, 5, len(agents))
        agents = np.clip(agents + growth, 0, None)

        # Population growth (Births)                            #Net Migration  =  some factor between 0 and and 10%
        new_agents = int(len(agents) * population_growth_rate) # change to a linear function .
        new_wealth = initialize_wealth(new_agents, alpha, x_min)
        agents = np.concatenate([agents, new_wealth])
        birth_counts.append(new_agents)

        # Apply death rate and inheritance tax
        death_mask = np.random.random(len(agents)) < death_rate
        deaths = np.sum(death_mask)
        death_rates_recorded.append(deaths / len(agents) if len(agents) > 0 else 0)
        inheritance = np.sum(apply_inheritance_tax(agents[death_mask], inheritance_tax_rate))
        inheritance_received.append(inheritance)
        agents = agents[~death_mask]

        # Distribute inheritance equally among surviving agents
        if len(agents) > 0:
            agents += inheritance / len(agents)

        # Calculate metrics
        gini = gini_coefficient(agents)
        gini_coefficients.append(gini)
        top_10_percent_index = int(len(agents) * 0.9)
        top_10_share_value = np.sum(np.partition(agents, -top_10_percent_index)[-top_10_percent_index:]) / np.sum(agents)
        top_10_share.append(top_10_share_value)
        population_sizes.append(len(agents))
        death_counts.append(deaths)

        # Wealth distribution by decile
        decile_distribution = calculate_wealth_distribution(agents)
        wealth_distribution_by_decile.append(decile_distribution)

        # GDP per capita
        total_wealth = np.sum(agents)
        gdp = total_wealth / len(agents) if len(agents) > 0 else 0
        gdp_per_capita.append(gdp)

        # Growth rate
        growth_rate = np.mean(growth) if len(growth) > 0 else 0
        growth_rates.append(growth_rate)

        #Assign each average to each decile to a decile

        #for each p in P(T) we can map some wealth w

        #define average wealth per decile (use some dummy numbers){T=0}

        #Define average birth rate per decile (uniform)

        #Deifine average death rate per decile(uniform)

        #Net migration petween 0,0.1 of total population (Implement a trump effect of migration )

        #population per decile = P(t) = p(t-1) + B(t,t-1)- d(T-1,t) + x.P(t-1)(how the population is. operation in a time period)

        #total population is sum of population

        #total wealth per decile per time period is average wealth of decile into population of decile


#society Rules(Intial condition)
      # Trade between Deciles
      # cap trade between decile to between neighbourhoods to be higher than others.
      #outcome of trade we will random between 0 and avg wealth trade decile(who you trading you).

      #design a parameter that caps(20%) of the outcome of trade gets added to your wealth in that time period.

      # at each time period calculate each individual all over again. (Make it in terms of a table.)




    return (
        gini_coefficients, top_10_share, final_wealth, population_sizes, death_counts,
        birth_counts, inheritance_received, gdp_per_capita, wealth_distribution_by_decile,
        growth_rates, death_rates_recorded
    )


Displaying Dummy Data across Intial time Periods

In [6]:
df_agents = initialize_wealth(Total_Population, wealth_per_decile, birth_rate, death_rate, net_migration)

print(df_agents)

       ID  Decile  Wealth  Initial Wealth (T=0)  Birth Rate  Death Rate  \
0       1       1     100                   100         0.1      0.0250   
1       2       1     100                   100         0.1      0.0250   
2       3       1     100                   100         0.1      0.0250   
3       4       1     100                   100         0.1      0.0250   
4       5       1     100                   100         0.1      0.0250   
..    ...     ...     ...                   ...         ...         ...   
995   996      10    1000                  1000         0.2      0.0005   
996   997      10    1000                  1000         0.2      0.0005   
997   998      10    1000                  1000         0.2      0.0005   
998   999      10    1000                  1000         0.2      0.0005   
999  1000      10    1000                  1000         0.2      0.0005   

     Net Migration  
0             0.05  
1             0.05  
2             0.05  
3             0

Update Population

In [15]:
import numpy as np
import pandas as pd

def update_population(df, birth_rate, death_rate, net_migration, wealth_per_decile):
    """
    Updates the agent population by considering births, deaths, and net migration per decile.

    Parameters:
    - df (pd.DataFrame): DataFrame containing agents' wealth and decile information.
    - birth_rate (list): Birth rate per decile.
    - death_rate (list): Death rate per decile.
    - net_migration (list): Net migration per decile.
    - wealth_per_decile (list): Average wealth values per decile.

    Returns:
    - Updated DataFrame with adjusted population.
    """
    df = df.copy()
    decile_counts = df['Decile'].value_counts().to_dict()
    new_agents = []

    for decile in range(len(birth_rate)):  # Iterate over deciles (0 to 9)
        count = decile_counts.get(decile, 0)  # Get current count or 0 if no agents
        print(birth_rate[decile])
        # Calculate new births, deaths, and migrations
        births = int(count * birth_rate[decile])
        print(births)
        deaths = int(count * death_rate[decile])
        migration = int(count * net_migration[decile])

        # Update total expected count for this decile
        new_count = count + births - deaths + migration
        new_count = max(new_count, 0)  # Prevent negative population

        if new_count > count:
            # Number of agents to add
            agents_to_add = new_count - count

            # Sample wealth values from existing agents in the decile
            if count > 0:
                sampled_wealth = np.random.choice(df[df['Decile'] == decile]['Wealth'], size=agents_to_add, replace=True)
            else:
                sampled_wealth = [wealth_per_decile[decile]] * agents_to_add  # Fallback wealth

            # Create new agents with sampled wealth values
            new_entries = pd.DataFrame({'Wealth': sampled_wealth, 'Decile': [decile] * agents_to_add})
            new_agents.append(new_entries)

        elif new_count < count:
            # Randomly remove excess agents
            drop_indices = df[df['Decile'] == decile].sample(n=(count - new_count)).index
            df = df.drop(drop_indices)

    # Add new agents to the DataFrame
    if new_agents:
        df = pd.concat([df] + new_agents, ignore_index=True)

    return df

# Example usage
df_agents = update_population(df_agents, birth_rate, death_rate, net_migration, wealth_per_decile)

0.1
0
0.08
24
0.07
18
0.065
16
0.05
103
0.048
9
0.04
7
0.03
4
0.022
2
0.2
191


Simultating Trade{Random Pairing}: Simplest


In [8]:

#a utility function. Each agent has a utility from wealth, and trade occurs if both can increase their utility. For example, Agent A has wealth W_a, Agent B has W_b. They exchange some delta, such that U(W_a + delta) > U(W_a) and U(W_b - delta) > U(W_b). But this requires defining a utility function, like logarithmic utility (diminishing returns), so that trades can be mutually beneficial.
import numpy as np

def simulate_trades(df, epsilon=0.05):
    """
    Simulate wealth exchange between randomly paired agents.

    Parameters:
    - df (pd.DataFrame): Agents DataFrame.
    - epsilon (float): Mixing parameter (0 < epsilon < 1).

    Returns:
    - Updated DataFrame with new wealth values.
    """
    df = df.copy()
    indices = df.index.to_numpy()
    np.random.shuffle(indices)  # Shuffle agents

    # Pair agents and perform exchange
    for i in range(0, len(indices) - 1, 2):
        idx1 = indices[i]
        idx2 = indices[i+1]

        w1 = df.at[idx1, 'Wealth']
        w2 = df.at[idx2, 'Wealth']

        # Update wealth using kinetic exchange rule #add rate of return
        new_w1 = (1 - epsilon) * w1 + epsilon * w2
        new_w2 = epsilon * w1 + (1 - epsilon) * w2

        df.at[idx1, 'Wealth'] = new_w1
        df.at[idx2, 'Wealth'] = new_w2

    return df

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [9]:

def simulate_trades_with_dynamic_prob(df, epsilon=0.05, savings_rate=0.2):
    """
    Simulates wealth exchange with savings and trade probability based on decile difference.

    Parameters:
    - df (pd.DataFrame): DataFrame with agents' wealth and deciles.
    - epsilon (float): Exchange fraction (0 < epsilon < 1).
    - savings_rate (float): Fraction of wealth saved before trading.

    Returns:
    - Updated DataFrame with new wealth values (rounded to two decimal places).
    """
    df = df.copy()
    indices = df.index.to_numpy()
    np.random.shuffle(indices)  # Shuffle agents randomly

    # Pair agents and perform exchange
    for i in range(0, len(indices) - 1, 2):
        idx1 = indices[i]
        idx2 = indices[i + 1]

        decile1 = df.at[idx1, 'Decile']
        decile2 = df.at[idx2, 'Decile']

        # Compute trade probability based on decile difference
        decile_diff = abs(decile1 - decile2)
        trade_prob = 1 / (decile_diff + 1)  # Ensure p = 1 for same decile, decreases otherwise

        # Skip trade with probability 1 - trade_prob
        if np.random.rand() > trade_prob:
            continue

        w1 = df.at[idx1, 'Wealth']
        w2 = df.at[idx2, 'Wealth']

        # Apply savings rate (agents only trade with remaining wealth)
        tradable_w1 = (1 - savings_rate) * w1
        tradable_w2 = (1 - savings_rate) * w2

        # Kinetic exchange model update
        new_tradable_w1 = (1 - epsilon) * tradable_w1 + epsilon * tradable_w2
        new_tradable_w2 = epsilon * tradable_w1 + (1 - epsilon) * tradable_w2

        # Final wealth = saved wealth + updated tradable wealth (rounded)
        df.at[idx1, 'Wealth'] = round((savings_rate * w1) + new_tradable_w1, 2)
        df.at[idx2, 'Wealth'] = round((savings_rate * w2) + new_tradable_w2, 2)

    return df

In [10]:
def update_deciles(df):
    """
    Reassign deciles based on current wealth.
    """
    df = df.sort_values('Wealth', ascending=False)
    df['Decile'] = pd.qcut(df['Wealth'], q=10, labels=False) + 1
    return df

#df_agents = update_deciles(df_agents)

In [11]:
df_agents = simulate_trades_with_dynamic_prob(df_agents, epsilon=0.05, savings_rate=0.2)

# Display first 10 agents
print("Intial Trades")
print(df_agents.head(10))

Intial Trades
     ID  Decile  Wealth  Initial Wealth (T=0)  Birth Rate  Death Rate  \
0   1.0       1     128                 100.0         0.1       0.025   
1   2.0       1     100                 100.0         0.1       0.025   
2   3.0       1     100                 100.0         0.1       0.025   
3   4.0       1     112                 100.0         0.1       0.025   
4   5.0       1     100                 100.0         0.1       0.025   
5   6.0       1     100                 100.0         0.1       0.025   
6   7.0       1     100                 100.0         0.1       0.025   
7   8.0       1     100                 100.0         0.1       0.025   
8   9.0       1     100                 100.0         0.1       0.025   
9  10.0       1     100                 100.0         0.1       0.025   

   Net Migration  
0           0.05  
1           0.05  
2           0.05  
3           0.05  
4           0.05  
5           0.05  
6           0.05  
7           0.05  
8          

In [12]:
for t in range(10):
    df_agents = simulate_trades_with_dynamic_prob(df_agents, epsilon=0.05)
    # Example usage
    df_agents = update_population(df_agents, birth_rate, death_rate, net_migration, wealth_per_decile)
    print(len(df_agents))
    df_agents.to_csv(f"df_agents_t{t}.csv", index=False)  # Save each table with a unique filename

1204
1331


<ipython-input-9-122e4c39a1e2>:45: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '903.84' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[idx1, 'Wealth'] = round((savings_rate * w1) + new_tradable_w1, 2)


1481
1658
1872
2130
2444
2828
3304
3889


In [13]:
# Display first 10 agents
print("Trade after 10 Time Steps ")
print(len(df_agents))
print(df_agents.groupby('Decile')['Wealth'].mean())

Trade after 10 Time Steps 
3889
Decile
1     128.335580
2     221.855679
3     318.184641
4     408.125361
5     498.488681
6     593.613647
7     690.860473
8     783.305338
9     875.143791
10    966.451000
Name: Wealth, dtype: float64


Simulating Trade{Wealth-Based Trade}

Simulating Trade{Supple and Demand}

Simulationg{Market Mechanism}


Define a Market Mechanism


Agent Based Behaviour in Trade

Transaction Rules


Wealth Redistribution Through Trade

Evaluaton
